# D3-07 Kaggle QLoRA / PEFT Tuning Notebook

**Purpose:** run the PEFT/QLoRA part on Kaggle GPU so the project has a real tuned-adapter row instead of `N/A - adapter not trained`.

This notebook expects the project files to be available in Kaggle either by:

1. uploading a Kaggle Dataset/ZIP that contains `data/tuning/finetune_qa.jsonl`, or
2. cloning the GitHub repo if internet is enabled.

**Outputs created by this notebook:**

- `outputs/qlora_adapter/` ? trained LoRA/QLoRA adapter files.
- `outputs/d3_or_final_zero_shot_vs_tuned.csv` ? replacement comparison table with real tuned metrics.
- `outputs/d3_tuning_latency.csv` ? real training/evaluation latency table.
- `outputs/qlora_training_summary.json` ? training card details.

Important: with only 15 training rows, this is a small demonstration adapter. Report it as a D3 PEFT/QLoRA feasibility/demo run, not as a robust climate domain model.


In [ ]:
# 0. Kaggle setup mode
# These are the exact Kaggle input paths provided by Aya.
from pathlib import Path
import os, json, time, math, shutil, subprocess, sys

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
OUTPUT_DIR = KAGGLE_WORKING / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_TABLES = KAGGLE_WORKING / 'reports' / 'tables'
REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

# Exact dataset paths from Kaggle.
KAGGLE_TUNING_PATH = Path('/kaggle/input/datasets/aayaehab/finetune-qa/finetune_qa.jsonl')
KAGGLE_BASELINE_TABLE_PATH = Path('/kaggle/input/datasets/aayaehab/d3-or-final-zero-shot-vs-tuned/d3_or_final_zero_shot_vs_tuned.csv')

# Fallback only if the exact paths are not available.
REPO_URL = 'https://github.com/Aya15155/climate_evidence_graphrag_agent.git'
PROJECT_DIR_NAME = 'climate_evidence_graphrag_agent'

print('Kaggle input:', KAGGLE_INPUT)
print('Kaggle working:', KAGGLE_WORKING)
print('Output dir:', OUTPUT_DIR)
print('Reports/tables dir:', REPORTS_TABLES)
print('Expected tuning path:', KAGGLE_TUNING_PATH)
print('Expected baseline table path:', KAGGLE_BASELINE_TABLE_PATH)


In [ ]:
# 1. Install required packages
# Kaggle images change, so install explicitly. This cell needs Kaggle internet ON.
import sys, subprocess

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

pip_install(
    'transformers>=4.42.0',
    'datasets>=2.19.0',
    'accelerate>=0.30.0',
    'peft>=0.11.0',
    'trl>=0.9.0',
    'bitsandbytes>=0.43.0',
    'evaluate>=0.4.2',
    'rouge-score>=0.1.2',
    'sentencepiece>=0.2.0',
)
print('Packages installed/importable.')


In [ ]:
# 2. Check GPU and libraries
import torch, importlib.util, platform

GPU_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'NO CUDA GPU'
print('Torch:', torch.__version__)
print('CUDA available:', GPU_AVAILABLE)
print('GPU:', GPU_NAME)
print('peft:', importlib.util.find_spec('peft') is not None)
print('bitsandbytes:', importlib.util.find_spec('bitsandbytes') is not None)
print('trl:', importlib.util.find_spec('trl') is not None)

if not GPU_AVAILABLE:
    raise RuntimeError('Kaggle GPU is not enabled. Go to Notebook settings -> Accelerator -> GPU, then restart and rerun.')


In [ ]:
# 3. Locate project files from the exact Kaggle input paths
# Read inputs directly from /kaggle/input. Write outputs only to /kaggle/working.
from pathlib import Path
import subprocess, os, json

# Exact input files provided by Aya.
TUNING_PATH = Path('/kaggle/input/datasets/aayaehab/finetune-qa/finetune_qa.jsonl')
BASELINE_TABLE = Path('/kaggle/input/datasets/aayaehab/d3-or-final-zero-shot-vs-tuned/d3_or_final_zero_shot_vs_tuned.csv')

# PROJECT_ROOT is only a context variable here. It is not used to find the tuning file.
PROJECT_ROOT = TUNING_PATH.parent

print('Reading tuning file directly from:', TUNING_PATH)
print('Reading baseline table directly from:', BASELINE_TABLE)
print('TUNING_PATH exists:', TUNING_PATH.exists())
print('BASELINE_TABLE exists:', BASELINE_TABLE.exists())

assert TUNING_PATH.exists(), (
    'Missing finetune_qa.jsonl at the exact Kaggle input path. '
    'Check that the finetune-qa dataset is attached to this Kaggle notebook. '
    f'Expected: {TUNING_PATH}'
)

if not BASELINE_TABLE.exists():
    print('WARNING: baseline table not found. Training can still run, but comparison context is missing.')


In [ ]:
# 4. Load and validate tuning rows directly from the exact Kaggle input path
# QLoRA training only needs finetune_qa.jsonl. It does NOT need sample_chunks.json.
import json, pandas as pd

required = {
    'instruction', 'input', 'output', 'source_document_id',
    'page_start', 'page_end', 'evidence_chunk_id', 'license_note'
}

assert TUNING_PATH.exists(), f'Missing required tuning file: {TUNING_PATH}'
rows = [json.loads(line) for line in TUNING_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
assert rows, 'finetune_qa.jsonl is empty.'

problems = []
for i, row in enumerate(rows, start=1):
    missing = sorted(required - set(row))
    if missing:
        problems.append({'row': i, 'missing_fields': missing})

if problems:
    raise ValueError(f'Some tuning rows are missing fields: {problems[:5]}')

train_df = pd.DataFrame(rows)
print('Loaded tuning rows:', len(train_df))
print('Source path:', TUNING_PATH)
display(train_df[['input', 'output', 'source_document_id', 'page_start', 'page_end', 'evidence_chunk_id']].head())


In [ ]:
# 5. Convert rows to instruction-tuning format
from datasets import Dataset

def format_example(row):
    # Keep page/chunk provenance in the prompt so the adapter learns citation-aware answering.
    return (
        '### Instruction:\n' + str(row['instruction']).strip() + '\n\n'
        '### Question:\n' + str(row['input']).strip() + '\n\n'
        '### Evidence metadata:\n'
        f"source_document_id: {row['source_document_id']}\n"
        f"page_range: {row['page_start']}-{row['page_end']}\n"
        f"evidence_chunk_id: {row['evidence_chunk_id']}\n\n"
        '### Answer:\n' + str(row['output']).strip()
    )

formatted = [{'text': format_example(r)} for r in rows]
dataset = Dataset.from_list(formatted)
# Tiny dataset: use all rows for training and all rows again for a small sanity evaluation.
train_dataset = dataset
eval_dataset = dataset
print(dataset[0]['text'][:1000])


In [ ]:
# 6. Model choice
# Good Kaggle choices:
# - TinyLlama/TinyLlama-1.1B-Chat-v1.0: safer on T4, fast demo.
# - microsoft/Phi-3-mini-4k-instruct: stronger but may need more memory/time.
# Use TinyLlama by default because the dataset has only 15 rows and D3 needs proof of pipeline, not a large benchmark.

BASE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
ADAPTER_DIR = OUTPUT_DIR / 'qlora_adapter'
MERGED_DIR = OUTPUT_DIR / 'qlora_merged_optional'
print('BASE_MODEL:', BASE_MODEL)
print('ADAPTER_DIR:', ADAPTER_DIR)


In [ ]:
# 7. Load model with 4-bit quantization and prepare LoRA
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# 8. Train QLoRA adapter
from trl import SFTTrainer
from transformers import TrainingArguments

training_start = time.time()

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'training_runs'),
    num_train_epochs=8,              # tiny dataset; enough to show adapter can fit format
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy='no',
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=1024,
    args=training_args,
    packing=False,
)

train_result = trainer.train()
training_duration_sec = round(time.time() - training_start, 2)
print('Training duration sec:', training_duration_sec)
print(train_result)

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved adapter to:', ADAPTER_DIR)


In [ ]:
# 9. Simple generation helpers for zero-shot vs tuned comparison
import torch, time, statistics
from peft import PeftModel

def build_prompt(question):
    return (
        'You are a climate evidence assistant. Answer concisely and include source/page awareness when possible.\n\n'
        f'Question: {question}\n'
        'Answer:'
    )

def generate_answer(model_obj, question, max_new_tokens=160):
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors='pt').to(model_obj.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency_ms = (time.perf_counter() - t0) * 1000
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = text.split('Answer:', 1)[-1].strip()
    return answer, latency_ms

def token_f1(pred, ref):
    import re
    p = re.findall(r'[a-zA-Z0-9]+', str(pred).lower())
    r = re.findall(r'[a-zA-Z0-9]+', str(ref).lower())
    if not p or not r:
        return 0.0
    p_counts = {}
    for t in p: p_counts[t] = p_counts.get(t, 0) + 1
    overlap = 0
    for t in r:
        if p_counts.get(t, 0) > 0:
            overlap += 1
            p_counts[t] -= 1
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(r)
    return 2 * precision * recall / (precision + recall)

def citation_presence_score(answer):
    # Proxy only: does the answer mention page/source/citation style?
    a = str(answer).lower()
    return float(('page' in a) or ('source' in a) or ('document' in a) or ('chunk' in a))


In [ ]:
# 10. Evaluate base model and tuned adapter on the 15 QA rows
# This is a lightweight D3 evaluation. It is not a full human/RAGAS evaluation, but it gives real post-tuning numbers.
from transformers import AutoModelForCausalLM

questions = [r['input'] for r in rows]
references = [r['output'] for r in rows]

# Evaluate zero-shot base model in 4-bit.
base_eval_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_eval_model.eval()

base_answers, base_lats = [], []
for q in questions:
    ans, lat = generate_answer(base_eval_model, q)
    base_answers.append(ans)
    base_lats.append(lat)

# Attach adapter to a fresh base for tuned evaluation.
tuned_model = PeftModel.from_pretrained(base_eval_model, ADAPTER_DIR)
tuned_model.eval()

tuned_answers, tuned_lats = [], []
for q in questions:
    ans, lat = generate_answer(tuned_model, q)
    tuned_answers.append(ans)
    tuned_lats.append(lat)

def summarize(method, answers, latencies):
    f1s = [token_f1(a, r) for a, r in zip(answers, references)]
    citation_scores = [citation_presence_score(a) for a in answers]
    return {
        'method': method,
        'training_status': 'completed' if method == 'qlora_tuned_adapter' else 'completed_baseline_only',
        'dataset_rows': len(rows),
        # Proxy names kept to match project table. Be clear in report that these are lexical/provenance proxies.
        'faithfulness': round(float(sum(f1s) / len(f1s)), 4),
        'answer_relevance': round(float(sum(f1s) / len(f1s)), 4),
        'citation_correct': round(float(sum(citation_scores) / len(citation_scores)), 4),
        'mean_latency_ms': round(float(sum(latencies) / len(latencies)), 2),
        'p95_latency_ms': round(float(sorted(latencies)[max(0, math.ceil(0.95 * len(latencies)) - 1)]), 2),
        'model_or_adapter': str(ADAPTER_DIR) if method == 'qlora_tuned_adapter' else BASE_MODEL,
        'notes': 'Kaggle GPU QLoRA run; metrics are lightweight lexical/provenance proxies on the 15-row tuning/eval set.' if method == 'qlora_tuned_adapter' else 'Base model zero-shot on same 15-row QA set.',
    }

comparison_rows = [
    summarize('zero_shot_graphrag_baseline', base_answers, base_lats),
    summarize('qlora_tuned_adapter', tuned_answers, tuned_lats),
]
comparison_df = pd.DataFrame(comparison_rows)
comparison_path = OUTPUT_DIR / 'd3_or_final_zero_shot_vs_tuned.csv'
comparison_reports_path = REPORTS_TABLES / 'd3_or_final_zero_shot_vs_tuned.csv'
comparison_df.to_csv(comparison_path, index=False)
comparison_df.to_csv(comparison_reports_path, index=False)
display(comparison_df)
print('Saved:', comparison_path)
print('Saved compatibility copy:', comparison_reports_path)


In [ ]:
# 11. Save detailed generations for audit
details = []
for i, (q, ref, b, t) in enumerate(zip(questions, references, base_answers, tuned_answers), start=1):
    details.append({
        'row': i,
        'question': q,
        'reference_answer': ref,
        'zero_shot_answer': b,
        'tuned_answer': t,
        'zero_shot_token_f1': token_f1(b, ref),
        'tuned_token_f1': token_f1(t, ref),
    })
details_df = pd.DataFrame(details)
details_path = OUTPUT_DIR / 'd3_qlora_generation_details.csv'
details_df.to_csv(details_path, index=False)
display(details_df.head())
print('Saved:', details_path)


In [ ]:
# 12. Save tuning latency table and training summary
import importlib.util

latency_rows = [
    {
        'stage': 'environment_check',
        'status': 'completed',
        'duration_sec': 0,
        'hardware': GPU_NAME,
        'torch_version': torch.__version__,
        'cuda_available': bool(torch.cuda.is_available()),
        'peft_installed': importlib.util.find_spec('peft') is not None,
        'bitsandbytes_installed': importlib.util.find_spec('bitsandbytes') is not None,
        'trl_installed': importlib.util.find_spec('trl') is not None,
        'notes': 'Kaggle GPU environment.',
    },
    {
        'stage': 'tuning_dataset_build',
        'status': 'completed',
        'duration_sec': 0,
        'hardware': GPU_NAME,
        'torch_version': torch.__version__,
        'cuda_available': bool(torch.cuda.is_available()),
        'peft_installed': True,
        'bitsandbytes_installed': True,
        'trl_installed': True,
        'notes': f'Loaded {len(rows)} rows from finetune_qa.jsonl.',
    },
    {
        'stage': 'qlora_training',
        'status': 'completed',
        'duration_sec': training_duration_sec,
        'hardware': GPU_NAME,
        'torch_version': torch.__version__,
        'cuda_available': bool(torch.cuda.is_available()),
        'peft_installed': True,
        'bitsandbytes_installed': True,
        'trl_installed': True,
        'notes': f'Base model={BASE_MODEL}; LoRA r=16 alpha=32 dropout=0.05; epochs=8.',
    },
]
latency_df = pd.DataFrame(latency_rows)
latency_path = OUTPUT_DIR / 'd3_tuning_latency.csv'
latency_reports_path = REPORTS_TABLES / 'd3_tuning_latency.csv'
latency_df.to_csv(latency_path, index=False)
latency_df.to_csv(latency_reports_path, index=False)
display(latency_df)
print('Saved:', latency_path)
print('Saved compatibility copy:', latency_reports_path)

summary = {
    'base_model': BASE_MODEL,
    'adapter_dir': str(ADAPTER_DIR),
    'dataset_rows': len(rows),
    'epochs': 8,
    'learning_rate': 2e-4,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'quantization': '4-bit NF4 QLoRA',
    'hardware': GPU_NAME,
    'training_duration_sec': training_duration_sec,
    'metrics_note': 'Lightweight lexical/provenance proxy metrics; use human/RAGAS evaluation for final claims if time allows.',
}
summary_path = OUTPUT_DIR / 'qlora_training_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved:', summary_path)


In [ ]:
# 13. Zip outputs for download and copying back to the repo
zip_path = KAGGLE_WORKING / 'd3_qlora_outputs.zip'
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path).replace('.zip', ''), 'zip', OUTPUT_DIR)
print('Download this file from Kaggle output panel:')
print(zip_path)
print('After download, copy these files back into your repo:')
print('1) outputs/d3_or_final_zero_shot_vs_tuned.csv -> reports/tables/d3_or_final_zero_shot_vs_tuned.csv')
print('2) outputs/d3_tuning_latency.csv -> reports/tables/d3_tuning_latency.csv')
print('3) outputs/qlora_adapter/ -> models/qlora_adapter/ or outputs/qlora_adapter/')
print('4) outputs/qlora_training_summary.json -> reports/tables or reports/')
